In [16]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!pip install flask pyngrok pydub librosa soundfile tensorflow requests


In [20]:
!apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [21]:
# List processes using port 5000
!lsof -i :5000


COMMAND  PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 7602 root   51u  IPv4 240769      0t0  TCP *:5000 (LISTEN)


In [ ]:
!kill -9 7602


In [1]:
import os
import numpy as np
import librosa
import tensorflow as tf
from pydub import AudioSegment
# Load the model (NEW FORMAT)
MODEL_PATH = "/content/drive/MyDrive/model/voice_model_darija.keras"
model = tf.keras.models.load_model(MODEL_PATH)

labels = ["ch3al", "sini bchwiya", "sini bzarba", "tfi"]

def extract_features(file_path, sr=16000, n_mfcc=40, max_sec=2):
    audio, _ = librosa.load(file_path, sr=sr)
    audio, _ = librosa.effects.trim(audio, top_db=20)

    max_len = sr * max_sec
    if len(audio) < max_len:
        audio = np.pad(audio, (0, max_len - len(audio)))
    else:
        audio = audio[:max_len]

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc, hop_length=256, n_fft=512)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    combined = np.vstack([mfcc, delta, delta2])

    return combined[..., np.newaxis]  # shape: (120, T, 1)


print(model.summary())


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 118, 124, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 57, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 28, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 53760)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     6,881,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,801,482 (52.65 MB)

 Trainable params: 6,900,740 (26.32 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,900,742 (26.32 MB)

None


In [2]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import time

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        if 'file' not in request.files:
            return jsonify({'error': 'no file'}), 400

        file = request.files['file']
        temp_path = "temp.wav"
        file.save(temp_path)

        # Extract features with same parameters as training
        features = extract_features(temp_path, max_sec=2)
        features = np.expand_dims(features, axis=0)  # batch dimension

        # Predict
        preds = model.predict(features)
        label_idx = np.argmax(preds)
        label = labels[label_idx]
        confidence = float(np.max(preds))

        # Top 3
        top3_idx = preds[0].argsort()[-3:][::-1]
        top3 = [{"label": labels[i], "prob": float(preds[0][i])} for i in top3_idx]

        return jsonify({
            "prediction": label,
            "confidence": confidence,
            "top3": top3
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500



@app.route("/")
def home():
    return "API is running!"


In [3]:
def run_app():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=True,       # <-- shows full traceback in logs
        use_reloader=False  # <-- important for notebooks to prevent double-start
    )


thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

time.sleep(2)  # wait for Flask

# Start ngrok tunnel
ngrok.set_auth_token("3620toMGts6dF9yvlucrWEVRV7l_3WWRPkAhRYdfmhHzv1mE4")  # replace with your token
public_url = ngrok.connect(5000)
print("🔥 Public URL:", public_url.public_url)  # <-- use .public_url when sending requests


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🔥 Public URL: https://benign-georgene-connotive.ngrok-free.dev


In [23]:
import requests
from pydub import AudioSegment

# Convert audio
audio_path = "/content/drive/MyDrive/z1212.ogg"
audio_wav = "/content/drive/MyDrive/z1212.wav"
AudioSegment.from_file(audio_path)\
    .set_frame_rate(16000)\
    .set_channels(1)\
    .set_sample_width(2)\
    .export(audio_wav, format="wav")

# Send request
url = public_url.public_url + "/predict"
with open(audio_wav, "rb") as f:
    files = {"file": f}
    r = requests.post(url, files=files)

print("Status:", r.status_code)
print("Response:", r.json())


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step


INFO:werkzeug:127.0.0.1 - - [28/Nov/2025 01:22:56] "POST /predict HTTP/1.1" 200 -


Status: 200
Response: {'confidence': 0.9165323376655579, 'prediction': 'sini bzarba', 'top3': [{'label': 'sini bzarba', 'prob': 0.9165323376655579}, {'label': 'sini bchwiya', 'prob': 0.08345936983823776}, {'label': 'ch3al', 'prob': 6.644455424975604e-06}]}
